# Readmission Analysis

This notebook examines how recorded prior service use and information from the representative inpatient encounter relate to Any Readmission. The intended prediction point is near discharge.

The analysis uses the full prepared cohort, including patients assigned to Test. The cohort results describe historical records. Document any influence they have on later feature, rule, threshold, or model choices. Those choices are evaluated on the development data under the modeling protocol.

## 1. Cohort and Outcome

Read the prepared release and its target mapping. The positive class `readmitted` combines `<30` and `>30`. The negative class `not_readmitted` comes from `NO`, which indicates no readmission recorded within the source's observation coverage. Each patient contributes one eligible encounter selected by the smallest eligible numeric encounter ID.

In [ ]:
import sys
from pathlib import Path

from IPython.display import display

working_directory = Path.cwd().resolve()
project_root = next(
    directory
    for directory in (working_directory, *working_directory.parents)
    if (directory / "src" / "paths.py").is_file() and (directory / "requirements.txt").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.analysis.data import load_analysis_inputs

analysis = load_analysis_inputs(project_root=project_root)

In [ ]:
print(f"Prepared cohort: {len(analysis.target):,} patients and {analysis.predictors.shape[1]} predictors.")
display(analysis.target.value_counts(sort=False).rename("patients").to_frame())

## 2. Statistical Conventions

Let N be the cohort size, Y the total number of recorded readmissions, n the number of patients in a group, and y its recorded readmissions.

| Measure | Calculation |
| --- | --- |
| Coverage | n / N |
| Within-group readmission rate | y / n |
| Share of the cohort's recorded readmissions | y / Y |
| Rule support | y / N |
| Rule confidence | y / n |
| Rule lift | (y / n) / (Y / N) |

A denominator of zero makes the corresponding measure undefined. Tables flag groups below 30 patients or below 5 patients in either outcome class. These thresholds guide interpretation.

## 3. Q1. Prior Service Use

Compare patients with and without recorded outpatient, emergency, or inpatient visits in the previous year. Use the same groups to count how many readmissions a fixed rule based on any recorded prior visit captures or misses.

### History groups and recorded readmissions

A recorded count above zero indicates prior service use. When any of the three counts is missing, the joint history is unknown. Coverage uses the full cohort as its denominator. The readmission rate uses patients in the group, while the share of recorded readmissions uses all positive cases in the cohort.

In [ ]:
from matplotlib import pyplot as plt

from src.analysis.plots import plot_prior_visit_summary
from src.analysis.prior_visits import profile_prior_visits

prior_visits = profile_prior_visits(analysis)
display(prior_visits.prior_visit_summary.round(2))

figure = plot_prior_visit_summary(prior_visits.prior_visit_summary)
plt.close(figure)
display(figure)

### Service combinations and prior inpatient counts

The combination table includes all eight patterns for the three services. The inpatient table uses the count for that service alone and groups it as 0, 1, or 2 or more.

In [ ]:
display(prior_visits.prior_visit_combinations.round(2))
display(prior_visits.prior_inpatient_levels.round(2))

### Fixed alert based on prior service use

The rule alerts when at least one prior visit is recorded across the three services. The confusion matrix uses patients with complete history counts. Precision uses alerts as its denominator. Recall uses recorded readmissions among evaluated patients.

In [ ]:
display(prior_visits.history_rule_confusion_matrix.round(2))

**Observation.** Patients with a recorded prior visit had a readmission rate of 54.10% (9,849 of 18,206). Patients with all three recorded counts at zero had a rate of 36.04% (18,662 of 51,784). The latter group contained 65.46% of all recorded readmissions (18,662 of 28,511).

**Implication.** The fixed rule flagged 18,206 patients. It identified 9,849 recorded readmissions and flagged 8,357 patients with no recorded readmission. Its precision was 54.10% and its recall was 34.54%. On the same evaluation patients, a later model can be compared by how many additional cases it identifies among those with zero recorded visits and how many additional false positives it produces.

**Limitation.** These counts describe the full historical cohort, including Test patients. The source counts cover recorded visits during the previous year. Comparing the rule with a model requires computing both sets of metrics on the same evaluation patients.

## 4. Q2. Current Encounter Characteristics

Compare readmission rates by length of stay, number of diagnoses, and first diagnosis group within each prior service group. Interpret encounter fields in the context of the planned prediction point near discharge. Their availability at that point requires verification.

### Length of stay within each history group

Compare stays of 1–3, 4–7, and 8–14 days. Each bar shows the readmission rate for one stay band and history group. The label above it gives the rate and its patient denominator. Missing or out-of-range stay values appear in the table when present.

In [ ]:
from src.analysis.current_encounters import profile_current_encounters
from src.analysis.plots import plot_stay_by_history

current_encounters = profile_current_encounters(analysis)
display(current_encounters.stay_by_history.round(2))

figure = plot_stay_by_history(current_encounters.stay_by_history)
plt.close(figure)
display(figure)

### Diagnosis count and first diagnosis group

The tables show each observed count and first diagnosis group in both history groups. An empty diagnosis count appears as a missing value. A blank first diagnosis group is shown as `Blank field`. The source grouping labels `Missing` and `Unmapped` retain their meanings. The `small_group` column identifies rates based on limited observations.

In [ ]:
display(current_encounters.diagnoses_count_by_history.round(2))
display(current_encounters.primary_diagnosis_by_history.round(2))

**Observation.** Among patients with no recorded prior visits, the readmission rate was 33.09% for stays of 1–3 days (8,671 of 26,203) and 41.37% for stays of 8–14 days (2,935 of 7,094), a difference of 8.28 percentage points. The corresponding rates among patients with a recorded prior visit were 51.17% (4,576 of 8,943) and 58.43% (1,563 of 2,675).

**Observation.** Within the group with zero recorded visits, 8,606 of 21,237 patients with nine diagnoses had a recorded readmission (40.52%). The first diagnosis groups `Circulatory` and `Neoplasms` had rates of 38.19% (6,228 of 16,309) and 29.12% (564 of 1,937), respectively. Each diagnosis count from 10 through 16 had fewer than 30 patients in this history group. Its `Other_E` first diagnosis group had one patient.

**Implication.** A later model comparison can test whether adding these encounter variables to the three history counts improves recall among patients with no recorded prior visits. Report false positives for the same patients at the selected operating threshold.

**Limitation.** These unadjusted differences describe the historical cohort. They can reflect differences in the patients who fall in each group. Model evaluation is needed to establish predictive gain.

## 5. Q3. Laboratory and Treatment Records

Measure laboratory result coverage and readmission rates by recording pattern. Preserve the source result levels and treatment statuses. The literal laboratory value `None` means no result is recorded. A result-level rate uses patients in that level of the corresponding test.

### Laboratory recording patterns

The four patterns classify whether each test has a recorded result. Coverage uses all patients in the cohort. The readmission rate uses the patients in the corresponding pattern. If a blank or unexpected value leaves either test's recording status unknown, the patient appears in the unknown pattern.

In [ ]:
from src.analysis.clinical_records import profile_clinical_records
from src.analysis.plots import plot_lab_recording_by_outcome

clinical_records = profile_clinical_records(analysis)
display(clinical_records.lab_recording_by_outcome.round(2))

figure = plot_lab_recording_by_outcome(clinical_records.lab_recording_by_outcome)
plt.close(figure)
display(figure)

### Recorded results by test

For each test, `percent_of_recorded_results` describes the distribution of the recorded levels using the number of patients with a result for that test. `readmission_rate_percent` uses the patients in the specific result level. The table also reports the literal `None`, blank fields, and unexpected values separately when they occur.

In [ ]:
display(clinical_records.lab_results_by_outcome.round(2))

### Treatment statuses

The table compares the source categories for `insulin`, `change`, and `diabetesMed`. Each column describes the same cohort, so the counts for different columns may include the same patients.

In [ ]:
display(clinical_records.treatment_by_outcome.round(2))

**Observation.** Neither result was recorded for 54,041 of 69,990 patients (77.21%). This group contained 22,206 recorded readmissions (41.09% of the group). The group with both results had a readmission rate of 55.69% (137 of 246) and covered 0.35% of the cohort.

**Observation.** HbA1c levels describe 12,846 patients with a recorded result, and glucose levels describe 3,349. Among patients with glucose recorded, the readmission rate was 49.86% for `>300` (355 of 712) and 42.15% for `Norm` (717 of 1,701). In the treatment table, the insulin `Down` and `No` groups had readmission rates of 45.32% (3,319 of 7,324) and 38.99% (13,361 of 34,268), respectively.

**Implication.** Later Weka evaluation can compare errors by recording pattern. A rule that requires both results can cover at most 246 patients in this cohort. Report the number of matched patients and recorded readmissions alongside rule confidence.

**Limitation.** The source records whether a result is present. Testing decisions, availability of results in practice, and differences in patient characteristics may influence which groups are observed. The treatment comparisons are descriptive and cannot establish treatment effects.

## 6. Q4. Combined Conditions

Compare each specified pair of conditions with its individual conditions. Report coverage, support, confidence, and lift with the underlying counts. Use these comparisons as references for the later Weka evaluation.

### Individual and combined conditions

For each pair, condition A describes prior service use and condition B is a stay of 8–14 days. The two A conditions are at least one prior inpatient visit and zero recorded visits across all three service types. Each row evaluates A, B, or their intersection as the antecedent of a rule with `readmitted` as the outcome.

Coverage and support both use the full cohort. Confidence uses confirmed antecedent matches. Lift compares confidence with the cohort readmission rate. The confidence differences compare the intersection with each component in percentage points. Incomplete records with unknown match status are counted separately.

In [ ]:
from src.analysis.rule_comparisons import compare_rule_conditions

rule_comparisons = compare_rule_conditions(analysis)
display(rule_comparisons.round(2))

**Observation.** The prior inpatient and 8–14 day stay combination had 60.77% confidence (959 of 1,578), compared with 60.31% for prior inpatient visits alone and 46.04% for the stay band alone. The gain over prior inpatient visits alone was 0.47 percentage points. The combination covered 2.25% of the cohort.

**Observation.** The zero prior visits and 8–14 day stay combination had 41.37% confidence (2,935 of 7,094). This was 5.33 percentage points above zero prior visits alone (36.04%) and 4.67 points below the stay band alone (46.04%).

**Implication.** When evaluating a rule with more than one condition in Weka, report how much confidence changes relative to each component and how many patients and recorded readmissions remain covered.

**Limitation.** Each antecedent is evaluated on the full historical cohort. Several rows can count the same patient, and a descriptive confidence difference can reflect the patients who match the combined conditions.

## 7. Weka Evaluation Reference

Use this table to carry the cohort findings, their limitations, and the comparison metrics into Weka. Run the comparisons on Validation after developing the model or rules on Train.

In [ ]:
from pandas import option_context

from src.analysis.evaluation import build_evaluation_map

evaluation_map = build_evaluation_map(analysis, prior_visits, current_encounters, clinical_records, rule_comparisons)
with option_context("display.max_colwidth", None, "display.max_columns", None):
    display(evaluation_map)